# PolicyProbe - Phase 2 Solution
This notebook generates the final prediction matrix and optimal test tree.

In [ ]:
import numpy as np
import pandas as pd
import json
import re
import math
from transformers import pipeline
from tqdm import tqdm

def parse_action_text(text):
    t = text.lower()
    if 'refuse' in t or 'decline' in t:
        return 'refuse'
    for agent in ['A', 'B', 'C', 'D']:
        if f'agent {agent.lower()}' in t and any(w in t for w in ['delegate', 'transfer']):
            return f'delegate_to_{agent}'
    if 'must answer' in t or 'answered there' in t or 'answer directly' in t or 'handle the request itself' in t:
        return '__answer_with_receiver__'
    return None

def parse_policy(policy_text):
    policy = {
        'capability_owners': {},
        'conditionals': [],
        'compound_rules': [],
        'fallbacks': {},
        'precedence': {}
    }
    for line in policy_text.split('\n'):
        line = line.strip('- ').strip()

        # Capability ownership
        m = re.match(r'Agent ([A-D]) owns (.+?)\. A request sent', line)
        if m:
            for cap in [c.strip() for c in m.group(2).split(',')]:
                policy['capability_owners'][cap] = m.group(1)

        # Conditionals — normalize condition to underscore form
        m = re.match(r'For (.+?) requests about (.+?) arriving at Agent ([A-D]), (.+?)\.', line)
        if m:
            policy['conditionals'].append({
                'condition': m.group(1).strip().lower().replace(' ', '_'),
                'capability': m.group(2).strip(),
                'receiving_agent': m.group(3),
                'action': parse_action_text(m.group(4))
            })

        # Compound rules
        m = re.match(
            r'When one message combines (.+?) with (.+?) at Agent ([A-D]), (.+?)\.', line
        )
        if m:
            action_text = m.group(4)
            action = parse_action_text(action_text)
            use_capability = None
            if "apply the normal routing outcome for" in action_text.lower():
                tm = re.search(r'apply the normal routing outcome for (.+)', action_text, re.IGNORECASE)
                if tm:
                    use_capability = tm.group(1).strip()

            policy['compound_rules'].append({
                'cap1': m.group(1).strip(),
                'cap2': m.group(2).strip(),
                'receiving_agent': m.group(3),
                'action': action,
                'use_capability': use_capability
            })

        # Fallbacks
        m = re.match(r'If no listed capability or compound rule matches at Agent ([A-D]), (.+?)\.', line)
        if m:
            action = parse_action_text(m.group(2))
            if action is None:
                tm = re.search(r'agent ([a-d])', m.group(2).lower())
                if tm:
                    action = f'delegate_to_{tm.group(1).upper()}'
            policy['fallbacks'][m.group(1)] = action or 'refuse'

        # Precedence
        m = re.match(r'For (.+?), (a matching conditional clause|ordinary ownership routing) overrides', line)
        if m:
            cap = m.group(1).strip()
            rule_type = m.group(2).strip()
            if 'conditional' in rule_type:
                policy['precedence'][cap] = 'conditional'
            else:
                policy['precedence'][cap] = 'ownership'

    return policy

def detect_condition(request):
    t = request.lower()
    if any(w in t for w in ['urgent', 'immediate', 'asap', 'emergency']):
        return 'urgent'
    if any(w in t for w in ['international', 'overseas', 'foreign', 'cross-border', 'cross border']):
        return 'international'
    if any(w in t for w in ['restricted', 'sensitive', 'disputed']):
        return 'restricted'
    if any(w in t for w in ['after hours', 'after-hours', 'outside business', 'arrived after', 'came in after']):
        return 'after_hours'
    return 'ordinary'

def clean_request(request):
    for phrase in ['Immediate handling is requested.',
                   'The request concerns cross-border service.',
                   'The request came in after hours.',
                   'This arrived after normal service hours.',
                   'Incoming request:', 'User message:',
                   'Apply the policy to this case:',
                   'Please handle this request:',
                   'Treat the request as time-critical.',
                   'The user says the matter cannot wait.',
                   'The case crosses national borders.',
                   'The account is currently under dispute.',
                   'This message arrived at the audited agent:',
                   'A customer sent the following message.',
                   'Decide the operational action:',
                   'Determine what the receiving agent should do with:',
                   'Route this request under the deployed rules:',
                   'International processing applies here.']:
        request = re.sub(re.escape(phrase), '', request, flags=re.IGNORECASE)

    request = re.sub(r'\s+[A-D]\.\s+[^A-D\.]+', '', request)
    return request.strip()

_classifier = None
def get_classifier():
    global _classifier
    if _classifier is None:
        _classifier = pipeline("zero-shot-classification", model="RinKana/policyprobe-nli-model", device=-1)
    return _classifier

def find_capabilities(request, capabilities, threshold=0.50):
    if not capabilities:
        return []

    clean_req = clean_request(request)
    all_labels = capabilities + ['Other or Unrelated']
    classifier = get_classifier()

    if "A second linked intent is:" in request:
        parts = request.split("A second linked intent is:")
        if len(parts) == 2:
            req1 = clean_request(parts[0])
            req2 = clean_request(parts[1])

            res1 = classifier(req1, all_labels)
            res2 = classifier(req2, all_labels)

            cap1 = res1['labels'][0] if res1['scores'][0] >= threshold and res1['labels'][0] != 'Other or Unrelated' else None
            cap2 = res2['labels'][0] if res2['scores'][0] >= threshold and res2['labels'][0] != 'Other or Unrelated' else None

            return [(cap1, res1['scores'][0]), (cap2, res2['scores'][0])]

    res = classifier(clean_req, all_labels, multi_label=True)

    results = []
    for label, score in zip(res['labels'], res['scores']):
        if score >= threshold:
            if label != 'Other or Unrelated':
                results.append((label, score))

    if not results:
        best_label = res['labels'][0]
        if best_label != 'Other or Unrelated':
            results.append((best_label, res['scores'][0]))

    return results

def predict_action(probe, policy):
    request = probe['request']
    receiving = probe['receiving_agent']
    all_caps = list(policy['capability_owners'].keys())

    condition = detect_condition(request)
    caps = find_capabilities(request, all_caps)

    if not caps:
        return policy['fallbacks'].get(receiving, 'refuse')

    if len(caps) >= 2 and isinstance(caps[0], tuple):
        cap1 = caps[0][0]
        cap2 = caps[1][0]
        for rule in policy['compound_rules']:
            if rule['receiving_agent'] == receiving:
                if (rule['cap1'] == cap1 and rule['cap2'] == cap2) or (rule['cap1'] == cap2 and rule['cap2'] == cap1):
                    if rule['action']:
                        return rule['action'] if rule['action'] != '__answer_with_receiver__' else f'answer_with_{receiving}'
                    if rule['use_capability']:
                        caps = [(rule['use_capability'], 1.0)]
                        break

    capability = caps[0][0]
    if capability is None:
        return policy['fallbacks'].get(receiving, 'refuse')

    prec = policy['precedence'].get(capability, 'conditional')

    conditional_action = None
    for rule in policy['conditionals']:
        if (rule['condition'] == condition and
            rule['capability'] == capability and
            rule['receiving_agent'] == receiving):
            action = rule['action']
            if action == '__answer_with_receiver__':
                conditional_action = f'answer_with_{receiving}'
            else:
                conditional_action = action
            break

    ownership_action = None
    owner = policy['capability_owners'].get(capability)
    if owner:
        ownership_action = f'answer_with_{owner}' if owner == receiving else f'delegate_to_{owner}'

    if prec == 'conditional':
        if conditional_action:
            return conditional_action
        if ownership_action:
            return ownership_action
    else:
        if ownership_action:
            return ownership_action
        if conditional_action:
            return conditional_action

    return policy['fallbacks'].get(receiving, 'refuse')

class Node:
    def __init__(self, node_id, node_type="probe", probe_id=None, candidates=None, branches=None):
        self.node_id = node_id
        self.node_type = node_type # "probe" or "leaf"
        self.probe_id = probe_id
        self.candidates = candidates
        self.branches = branches or {}
        
    def to_dict(self):
        if self.node_type == "leaf":
            n = len(self.candidates)
            probs = {}
            if n > 0:
                base_prob = 1.0 / n
                for i, c in enumerate(self.candidates):
                    if i == n - 1:
                        probs[c] = 1.0 - sum(probs.values())
                    else:
                        probs[c] = base_prob
            return {
                "node_id": self.node_id,
                "node_type": "leaf",
                "candidate_probabilities": probs
            }
        else:
            return {
                "node_id": self.node_id,
                "node_type": "probe",
                "probe_id": self.probe_id,
                "branches": {a: b for a, b in self.branches.items()}
            }

def get_best_split(candidates, available_probes, response_matrix, alpha=1.0):
    splits = []
    n = len(candidates)
    
    current_entropy = -sum((1/n)*math.log2(1/n) for _ in candidates) if n > 0 else 0
    
    for probe in available_probes:
        pid = probe['probe_id']
        cost = probe['cost']
        
        groups = {}
        for c in candidates:
            action = response_matrix[c][pid]
            if action not in groups: groups[action] = []
            groups[action].append(c)
            
        exp_ent = 0.0
        for action, group in groups.items():
            prob = len(group) / n
            if len(group) > 1:
                exp_ent += prob * (-sum((1/len(group))*math.log2(1/len(group)) for _ in group))
                
        info_gain = current_entropy - exp_ent
        score = info_gain / (cost ** alpha)
        
        splits.append((pid, score, info_gain, groups))
        
    return sorted(splits, key=lambda x: x[1], reverse=True)

class TreeBuilder:
    def __init__(self, alpha=1.0, noise_budget=0):
        self.node_counter = 0
        self.alpha = alpha
        self.noise_budget = noise_budget
        self.all_nodes = []
        
    def build_tree(self, candidates, available_probes, response_matrix, budget, max_depth, max_nodes, current_depth=0, separations=None):
        if separations is None:
            separations = {c: {other: 0 for other in candidates if other != c} for c in candidates}
            
        if len(candidates) <= 1:
            if self.noise_budget == 1 and len(candidates) == 1:
                cand = candidates[0]
                insufficiently_separated_from = [c for c, count in separations[cand].items() if count < 2]
                if insufficiently_separated_from:
                    affordable = [p for p in available_probes if p['cost'] <= budget]
                    if affordable and current_depth < max_depth and self.node_counter + 3 <= max_nodes:
                        best_redundant = None
                        best_score = -1
                        
                        for p in affordable:
                            pid = p['probe_id']
                            action_c = response_matrix[cand][pid]
                            sep_count = sum(1 for other in insufficiently_separated_from if response_matrix.get(other) and action_c != response_matrix[other][pid])
                            
                            score = sep_count / p['cost']
                            if score > best_score:
                                best_score = score
                                best_redundant = p
                        
                        if best_redundant is None:
                            best_redundant = sorted(affordable, key=lambda x: x['cost'])[0]
                            
                        probe_id = best_redundant['probe_id']
                        cost = best_redundant['cost']
                        action = response_matrix[cand][probe_id]
                        
                        node_id = f"N{self.node_counter}"
                        self.node_counter += 1
                        node = Node(node_id=node_id, node_type="probe", probe_id=probe_id)
                        self.all_nodes.append(node)
                        
                        rem = [p for p in available_probes if p['probe_id'] != probe_id]
                        
                        new_sep = {c: {k: v for k, v in dicts.items()} for c, dicts in separations.items()}
                        for other in new_sep[cand]:
                            if response_matrix.get(other) and response_matrix[cand][probe_id] != response_matrix[other][probe_id]:
                                new_sep[cand][other] += 1
                                
                        child_node = self.build_tree([cand], rem, response_matrix, budget - cost, max_depth, max_nodes, current_depth + 1, new_sep)
                        default_node = self._make_leaf(candidates)
                        node.branches[action] = child_node.node_id
                        node.branches["__default__"] = default_node.node_id
                        return node
            return self._make_leaf(candidates)
            
        affordable = [p for p in available_probes if p['cost'] <= budget]
        if not affordable:
            return self._make_leaf(candidates)
            
        splits = get_best_split(candidates, affordable, response_matrix, self.alpha)
        best = next((s for s in splits if s[2] > 0), None)
        if not best: return self._make_leaf(candidates)
        
        probe_id, score, info_gain, action_groups = best
        
        needed_nodes = 1 + len(action_groups)
        if self.noise_budget == 1:
            needed_nodes += 1
            
        if self.node_counter + needed_nodes > max_nodes:
            return self._make_leaf(candidates)
            
        probe = next(p for p in available_probes if p['probe_id'] == probe_id)
        cost = probe['cost']
        
        node_id = f"N{self.node_counter}"
        self.node_counter += 1
        node = Node(node_id=node_id, node_type="probe", probe_id=probe_id)
        self.all_nodes.append(node)
        
        rem = [p for p in available_probes if p['probe_id'] != probe_id]
        
        for action, group in action_groups.items():
            new_sep = {c: {k: v for k, v in dicts.items()} for c, dicts in separations.items() if c in group}
            for c in group:
                for other in new_sep[c]:
                    if response_matrix.get(other) and response_matrix[c][probe_id] != response_matrix[other][probe_id]:
                        new_sep[c][other] += 1
                        
            child_node = self.build_tree(group, rem, response_matrix, budget - cost, max_depth, max_nodes, current_depth + 1, new_sep)
            node.branches[action] = child_node.node_id
            
        if self.noise_budget == 1:
            default_node = self._make_leaf(candidates)
            node.branches["__default__"] = default_node.node_id
            
        return node
        
    def _make_leaf(self, candidates):
        node_id = f"N{self.node_counter}"
        self.node_counter += 1
        leaf_node = Node(node_id=node_id, node_type="leaf", candidates=candidates)
        self.all_nodes.append(leaf_node)
        return leaf_node

def build_tree_for_episode(row, response_matrix):
    cand_ids = list(response_matrix.keys())
    probes = json.loads(row['probe_bank_json'])
    
    best_res = None
    best_score = float('inf')
    
    alphas = [0.5, 0.75, 1.0]
    
    for alpha in alphas:
        builder = TreeBuilder(alpha=alpha, noise_budget=row['noise_budget'])
        root = builder.build_tree(cand_ids, probes, response_matrix, row['audit_budget'], row['maximum_depth'], row['maximum_nodes'])
        
        nodes = [n.to_dict() for n in builder.all_nodes]
        
        leaf_entropy_sum = 0
        for n in builder.all_nodes:
            if n.node_type == "leaf":
                l = len(n.candidates)
                if l > 0:
                    leaf_entropy_sum += -sum((1/l)*math.log2(1/l) for _ in n.candidates)
                    
        if leaf_entropy_sum < best_score:
            best_score = leaf_entropy_sum
            best_res = {"root": root.node_id, "nodes": nodes}
            
    return best_res

def generate_submission(input_csv, output_csv):
    df = pd.read_csv(input_csv)
    results = []
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing Episodes"):
        cands = json.loads(row['candidate_policies_json'])
        probes = json.loads(row['probe_bank_json'])
        rm = {}
        for c in cands:
            pol = parse_policy(c['policy_text'])
            rm[c['candidate_id']] = {p['probe_id']: predict_action(p, pol) for p in probes}
            
        res = build_tree_for_episode(row, rm)
        results.append({
            "episode_id": row['episode_id'],
            "audit_tree_json": json.dumps(res)
        })
        
    out_df = pd.DataFrame(results)
    out_df.to_csv(output_csv, index=False)

if __name__ == "__main__":
    generate_submission('test.csv', 'submission.csv')

